In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px




dados = pd.read_csv("dados_vendas.csv")

# 2. Média e mediana de preço por categoria
estatisticas = dados.groupby("Categoria")["Preco_Normal"].agg(
    ["mean", "median"]
)

def comparar_media_mediana(linha):
    if linha["mean"] > linha["median"]:
        return "Acima da mediana"
    elif linha["mean"] < linha["median"]:
        return "Abaixo da mediana"
    return "Igual à mediana"

estatisticas["Comparacao"] = estatisticas.apply(
    comparar_media_mediana, axis=1
)

print("\nEstatísticas por categoria:")
print(estatisticas)


# 3. Categoria com maior desvio padrão
desvio_categoria = dados.groupby("Categoria")["Preco_Normal"].std()
categoria_maior_desvio = desvio_categoria.idxmax()

dados_categoria = dados[
    dados["Categoria"] == categoria_maior_desvio
]


# 4. Boxplot e identificação de outliers
plt.figure(figsize=(8, 5))
plt.boxplot(dados_categoria["Preco_Normal"])
plt.title(f"Distribuição dos preços — {categoria_maior_desvio}")
plt.ylabel("Preço normal")
plt.tight_layout()
plt.show()

Q1 = dados_categoria["Preco_Normal"].quantile(0.25)
Q3 = dados_categoria["Preco_Normal"].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers = dados_categoria[
    (dados_categoria["Preco_Normal"] < limite_inferior) |
    (dados_categoria["Preco_Normal"] > limite_superior)
]

print("\nAnálise de outliers:")
print("Categoria analisada:", categoria_maior_desvio)
print("Quantidade de outliers:", len(outliers))
print(outliers[["title", "Preco_Normal"]])


# 5. Gráfico da média de desconto por categoria
media_desconto = dados.groupby("Categoria")["Desconto"].mean()

plt.figure(figsize=(10, 6))
plt.bar(media_desconto.index, media_desconto.values)
plt.title("Média de desconto por categoria")
plt.xlabel("Categoria")
plt.ylabel("Desconto médio")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# 6. Visualização interativa por categoria e marca
desconto_categoria_marca = (
    dados.groupby(["Categoria", "Marca"], as_index=False)["Desconto"]
    .mean()
)

fig = px.bar(
    desconto_categoria_marca,
    x="Categoria",
    y="Desconto",
    color="Marca",
    title="Média de desconto por categoria e marca",
    labels={
        "Categoria": "Categoria",
        "Desconto": "Desconto médio",
        "Marca": "Marca"
    },
    barmode="group"
)

fig.show()